# Frame Utilities - MERTIS Data Assembly

This module provides standalone functions for assembling, resampling, and analyzing MERTIS frame data.
Per **ADR-007**: Handles RAW vs CAL/PAR axis ordering differences.

## ADR-007: RAW vs CAL/PAR Axis Ordering

**Problem**: RAW level FITS files have axis order `(n_frames, n_pixels, n_spectrals)` while CAL/PAR have `(n_spectrals, n_pixels, n_frames)`.

**Solution**: Normalize RAW data on load by transposing to match CAL/PAR convention.

In [ ]:
#| export
#| default_exp frame_utils
import numpy as np
import pandas as pd
from typing import Optional
from scipy import ndimage

In [ ]:
#| export
def normalize_raw_axis_order(
    arr: np.ndarray,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> np.ndarray:
    """
    Normalize RAW level axis order to match CAL/PAR convention.

    Per ADR-007: RAW FITS files have shape (n_frames, n_pixels, n_spectrals)
    while CAL/PAR have (n_spectrals, n_pixels, n_frames).

    This function transposes RAW data from (temp, pix, spec) to (spec, pix, temp).

    Args:
        arr: Input array from RAW level FITS file
        spec_axis: Expected spectral axis in output (default: 0)
        pix_axis: Expected pixel axis in output (default: 1)
        temp_axis: Expected temporal axis in output (default: 2)

    Returns:
        Transposed array with order (spec, pix, temp)

    Example:
        >>> # RAW data comes as (62415, 100, 40) = (frames, pixels, spectrals)
        >>> raw_arr = np.random.randn(62415, 100, 40)
        >>> normalized = normalize_raw_axis_order(raw_arr)
        >>> normalized.shape  # (40, 100, 62415) = (spectrals, pixels, frames)
        (40, 100, 62415)
    """
    # RAW data comes in (temp_axis, pix_axis, spec_axis) order
    # We want to transpose to (spec_axis, pix_axis, temp_axis) = (0, 1, 2)
    # Current positions: temp=2, pix=1, spec=0 in the target
    # Source axes: [temp_axis, pix_axis, spec_axis] = [2, 1, 0]
    # Target: [0, 1, 2]
    # Transpose: move axis at position 2 to position 0, axis at 1 stays at 1, axis at 0 to position 2
    source_axes = [temp_axis, pix_axis, spec_axis]  # How axes are currently ordered in RAW
    # We want: spec at 0, pix at 1, temp at 2
    # Find where each source axis should go:
    # source_axes[0] (temp) should go to position 2
    # source_axes[1] (pix) should go to position 1
    # source_axes[2] (spec) should go to position 0
    transpose_order = [source_axes.index(i) for i in [spec_axis, pix_axis, temp_axis]]
    return np.transpose(arr, transpose_order)

In [ ]:
#| export
def get_frame_dimensions(
    frames_dict: dict,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> pd.DataFrame:
    """
    Analyze dimensions of all frames with standardized column order.

    Args:
        frames_dict: dict where values are arrays with 3D shape
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        DataFrame with columns: n_files, n_pixels, n_spectrals

    Example:
        >>> # Standard frame format (n_spectrals, n_pixels, n_temporal)
        >>> dims = get_frame_dimensions(frames, spec_axis=0, pix_axis=1, temp_axis=2)
        >>>
        >>> # Custom format (n_temporal, n_spectrals, n_pixels)
        >>> dims = get_frame_dimensions(frames, spec_axis=1, pix_axis=2, temp_axis=0)
    """
    return pd.DataFrame.from_dict(
        {
            k: {
                'n_files': v.shape[temp_axis],
                'n_pixels': v.shape[pix_axis],
                'n_spectrals': v.shape[spec_axis],
            }
            for k, v in frames_dict.items()
        },
        orient='index',
    )

In [ ]:
#| export
def detect_interpolation_target(
    frames_dict: dict,
    mode: str = 'match',
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> dict:
    """
    Determine target dimensions for frame assembly.

    Args:
        frames_dict: dict of 3D arrays per source file
        mode: 'match' (skip if uniform), 'up' (max dims), 'down' (min dims), 'none' (no interp)
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        dict with 'n_spectrals', 'n_pixels', 'mode_applied'

    Raises:
        ValueError: if mode='match' but dimensions are heterogeneous
    """
    if not frames_dict:
        raise ValueError("frames_dict is empty - cannot determine target dimensions")

    dims_df = get_frame_dimensions(frames_dict, spec_axis=spec_axis, pix_axis=pix_axis, temp_axis=temp_axis)

    # Check if all dimensions match
    spectrals_match = dims_df['n_spectrals'].nunique() == 1
    pixels_match = dims_df['n_pixels'].nunique() == 1

    if mode == 'match':
        if not (spectrals_match and pixels_match):
            raise ValueError(
                f"Mode='match' but heterogeneous dims found:\n{dims_df}\n"
                "Use mode='up' or 'down' for resampling."
            )
        return {
            'n_spectrals': dims_df['n_spectrals'].iloc[0],
            'n_pixels': dims_df['n_pixels'].iloc[0],
            'mode_applied': 'none',
            'reason': 'All dimensions uniform'
        }

    if mode == 'none':
        return {
            'n_spectrals': None,
            'n_pixels': None,
            'mode_applied': 'none',
            'reason': 'Interpolation disabled'
        }

    if mode == 'up':
        return {
            'n_spectrals': dims_df['n_spectrals'].max(),
            'n_pixels': dims_df['n_pixels'].max(),
            'mode_applied': 'up',
            'reason': 'Upsampling to largest frame'
        }

    if mode == 'down':
        return {
            'n_spectrals': dims_df['n_spectrals'].min(),
            'n_pixels': dims_df['n_pixels'].min(),
            'mode_applied': 'down',
            'reason': 'Downsampling to smallest frame'
        }

    raise ValueError(f"Unknown mode: {mode}. Choose from ['match', 'none', 'up', 'down']")

In [ ]:
#| export
def resample_frame(
    arr: np.ndarray,
    target_dims: dict,
    frame_key: str = '',
    order: int = 1,
    verbose: bool = True,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> tuple:
    """
    Resample a single frame to target dimensions using scipy.ndimage.zoom.

    Args:
        arr: Input array (3D)
        target_dims: dict with 'n_spectrals', 'n_pixels'
        frame_key: Name of frame for logging
        order: Interpolation order (1=linear, 3=cubic)
        verbose: Print resampling info
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        (resampled_array, interpolation_status_str)
    """
    current_shape = arr.shape
    n_spec = current_shape[spec_axis]
    n_pix = current_shape[pix_axis]
    n_files = current_shape[temp_axis]

    target_spec = target_dims['n_spectrals']
    target_pix = target_dims['n_pixels']

    # No resampling needed
    if (n_spec == target_spec) and (n_pix == target_pix):
        if verbose:
            print(f'  {frame_key}: shape {current_shape} → NO RESAMPLING')
        return arr, 'none'

    # Compute zoom factors preserving temporal axis
    zoom_factors = [1.0] * 3
    zoom_factors[spec_axis] = target_spec / n_spec
    zoom_factors[pix_axis] = target_pix / n_pix
    zoom_factors = tuple(zoom_factors)
    resampled = ndimage.zoom(arr, zoom_factors, order=order)

    if verbose:
        print(f'  {frame_key}: shape {current_shape} → {resampled.shape} '
              f'(zoom: {zoom_factors[0]:.2f}x, {zoom_factors[1]:.2f}x, order={order})')

    return resampled, f'zoom({zoom_factors[0]:.2f}x, {zoom_factors[1]:.2f}x, order={order})'

In [ ]:
#| export
def selective_resample_frames(
    frames_dict: dict,
    target_dims: dict,
    order: int = 1,
    verbose: bool = True,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> dict:
    """
    Resample frames that don't match target dimensions.
    Only interpolate when necessary.

    Args:
        frames_dict: dict of 3D arrays
        target_dims: dict with 'n_spectrals', 'n_pixels'
        order: Interpolation order (1=linear, 3=cubic)
        verbose: Print progress
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        dict of resampled arrays with same keys as input
    """
    if target_dims['n_spectrals'] is None:
        # No interpolation mode
        return frames_dict

    resampled_dict = {}

    if verbose:
        print(f"Resampling to target: n_spectrals={target_dims['n_spectrals']}, "
              f"n_pixels={target_dims['n_pixels']} (order={order})")

    for key, arr in frames_dict.items():
        resampled, status = resample_frame(arr, target_dims, frame_key=key, order=order, verbose=verbose,
                                          spec_axis=spec_axis, pix_axis=pix_axis, temp_axis=temp_axis)
        resampled_dict[key] = resampled

    return resampled_dict

In [ ]:
#| export
def assemble_frames(
    frames_dict: dict,
    sort_indices: np.ndarray = None,
    interp_mode: str = 'match',
    order: int = 1,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> tuple:
    """
    Core frame assembly function: interpolate heterogeneous frames and stack.

    Args:
        frames_dict: dict of 3D arrays
        sort_indices: Optional indices to reorder temporal axis (e.g., by TIME_UTC)
        interp_mode: Interpolation strategy
            - 'match': No interpolation, require uniform dims (safe)
            - 'up': Upsample all to largest frame size
            - 'down': Downsample all to smallest frame size
            - 'none': Skip interpolation (stack as-is, may fail if dims differ)
        order: Interpolation order (1=linear, 3=cubic)
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        (assembled_cube, metadata_dict) where:
        - assembled_cube: 3D array (concatenated temporally)
        - metadata_dict: dict with assembly info and parameters

    Raises:
        ValueError: if interp_mode='match' and dimensions are heterogeneous
    """
    # Determine target dimensions based on mode
    target_dims = detect_interpolation_target(frames_dict, mode=interp_mode,
                                             spec_axis=spec_axis, pix_axis=pix_axis, temp_axis=temp_axis)

    # Apply selective resampling if needed
    if target_dims['n_spectrals'] is not None:
        resampled_dict = selective_resample_frames(frames_dict, target_dims, order=order, verbose=False,
                                                   spec_axis=spec_axis, pix_axis=pix_axis, temp_axis=temp_axis)
    else:
        resampled_dict = frames_dict

    # Stack all frames along temporal axis (respects custom axis)
    # Keep track of file keys and their temporal indices for provenance
    sorted_keys = sorted(resampled_dict.keys())
    file_indices = []
    cumsum = 0
    for key in sorted_keys:
        n_frames = resampled_dict[key].shape[temp_axis]
        file_indices.append((cumsum, cumsum + n_frames))
        cumsum += n_frames

    stacked = np.concatenate([resampled_dict[k] for k in sorted_keys], axis=temp_axis)

    # Apply temporal sorting if provided
    if sort_indices is not None:
        slices = [slice(None)] * 3
        slices[temp_axis] = sort_indices
        stacked = stacked[tuple(slices)]

    # Build metadata with file provenance
    metadata = {
        'interpolation_mode': target_dims['mode_applied'],
        'interpolation_order': order,
        'target_n_spectrals': target_dims['n_spectrals'],
        'target_n_pixels': target_dims['n_pixels'],
        'n_temporal_samples': stacked.shape[temp_axis],
        'reason': target_dims.get('reason', ''),
        'file_keys': sorted_keys,
        'file_indices': file_indices,
    }

    return stacked, metadata

In [ ]:
#| export
def assemble_geometry(
    geom_dict: dict,
    target_dims: dict,
    sort_indices: np.ndarray = None,
    interp_mode: str = 'match',
    order: int = 1,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> dict:
    """
    Assemble geometry data to match science datacube dimensions.

    Args:
        geom_dict: dict of geometry arrays per variable (already stacked)
        target_dims: dict with 'n_spectrals', 'n_pixels' from science assembly
        sort_indices: Optional temporal sort indices (same as science)
        interp_mode: Must match science assembly mode
        order: Interpolation order
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        dict[variable_name] = 3D geometry array matching science cube shape
    """
    resampled_geom = {}

    for var_name, arr in geom_dict.items():
        if target_dims['n_spectrals'] is None:
            # No interpolation
            resampled_geom[var_name] = arr
        else:
            # Resample to match science dims
            resampled, _ = resample_frame(arr, target_dims, frame_key=var_name, order=order, verbose=False,
                                         spec_axis=spec_axis, pix_axis=pix_axis, temp_axis=temp_axis)
            resampled_geom[var_name] = resampled

            slices = [slice(None)] * 3
            slices[temp_axis] = sort_indices
            resampled_geom[var_name] = resampled_geom[var_name][tuple(slices)]


        return resampled_geom
    return resampled_geom

In [ ]:
#| export
def filter_frames_by_size(
    frames_dict: dict,
    min_frames: int = None,
    min_pixels: int = None,
    min_spectrals: int = None,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> dict:
    """
    Filter frames dict by minimum dimension thresholds.

    Useful for excluding scientifically irrelevant snippets (e.g., 3-frame acquisitions)
    before expensive upsampling operations.

    Args:
        frames_dict: dict of 3D arrays
        min_frames: Minimum temporal samples to keep (default: no filter)
        min_pixels: Minimum spatial pixels to keep (default: no filter)
        min_spectrals: Minimum spectral channels to keep (default: no filter)
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        Filtered dict with only frames meeting all thresholds

    Example:
        >>> # Exclude 3-frame snippets before upsampling
        >>> filtered = filter_frames_by_size(raw_frames, min_frames=100, spec_axis=0, pix_axis=1, temp_axis=2)
        >>> cube, meta = assemble_frames(filtered, interp_mode='up')
    """
    filtered = {}

    for key, arr in frames_dict.items():
        n_spec = arr.shape[spec_axis]
        n_pix = arr.shape[pix_axis]
        n_files = arr.shape[temp_axis]
        if min_pixels is not None and n_pix < min_pixels:
            continue
        if min_frames is not None and n_files < min_frames:
            continue

        filtered[key] = arr

    return filtered

In [ ]:
#| export
def get_frames_by_shape_groups(
    frames_dict: dict,
    metadata_df: pd.DataFrame = None,
    sort_by_time: bool = True,
    spec_axis: int = 0,
    pix_axis: int = 1,
    temp_axis: int = 2
) -> dict:
    """
    Group frames by spatial/spectral shape without interpolation.

    Returns dict of {(n_pixels, n_spectrals): (cube, metadata, file_keys)}
    for fast exploration of heterogeneous data without upsampling overhead.

    Args:
        frames_dict: dict of 3D arrays
        metadata_df: Optional metadata for filtering/sorting (must have 'file_index' column)
        sort_by_time: Sort each group's temporal axis by TIME_UTC (default: True)
        spec_axis: axis index for spectral dimension (default: 0)
        pix_axis: axis index for spatial/pixel dimension (default: 1)
        temp_axis: axis index for temporal dimension (default: 2)

    Returns:
        dict mapping shape_key -> dict with:
            - 'cube': stacked 3D array
            - 'metadata': dict with group stats
            - 'file_keys': list of source file keys in this group
            - 'frame_indices': per-file frame counts (useful for metadata slicing)

    Example:
        >>> groups = get_frames_by_shape_groups(raw_frames, ms.mertis_tis_metadata_df, spec_axis=0, pix_axis=1, temp_axis=2)
        >>> for shape_key, group_data in groups.items():
        ...     print(f"Shape {shape_key}: {group_data['cube'].shape}")
        ...     print(f"  Files: {group_data['metadata']['n_files']}")
        ...     print(f"  Total frames: {group_data['metadata']['n_total_frames']}")
    """
    from collections import defaultdict

    # Group by spatial/spectral dims (ignore temporal axis)
    shape_groups = defaultdict(list)

    for key, arr in frames_dict.items():
        n_spec = arr.shape[spec_axis]
        n_pix = arr.shape[pix_axis]
        n_files = arr.shape[temp_axis]
        shape_key = (n_pix, n_spec)  # (spatial, spectral) for readability
        shape_groups[shape_key].append((key, arr))

    # Assemble each group
    result = {}

    for shape_key, file_list in shape_groups.items():
        file_keys = [item[0] for item in file_list]
        arrays = [item[1] for item in file_list]

        # Stack along temporal axis (respects custom axis)
        stacked = np.concatenate(arrays, axis=temp_axis)

        # Build frame indices for metadata slicing
        frame_indices = []
        cumsum = 0
        for arr in arrays:
            n_frames = arr.shape[temp_axis]
            frame_indices.append((cumsum, cumsum + n_frames))
            cumsum += n_frames

        # Sort by time if metadata provided
        if sort_by_time and metadata_df is not None:
            # Extract relevant metadata rows for this group's files
            group_meta = metadata_df[metadata_df.index.isin(
                [i for start, end in frame_indices for i in range(start, end)]
            )]

            if 'TIME_UTC' in group_meta.columns and len(group_meta) == stacked.shape[temp_axis]:
                sort_idx = group_meta.sort_values('TIME_UTC').index.values
                # Create slice tuple with temporal sorting
                slices = [slice(None)] * 3
                slices[temp_axis] = sort_idx
                stacked = stacked[tuple(slices)]

        # Build metadata
        metadata = {
            'n_files': len(file_list),
            'n_total_frames': stacked.shape[temp_axis],
            'n_pixels': shape_key[0],
            'n_spectrals': shape_key[1],
            'shape_key': shape_key,
        }

        result[shape_key] = {
            'cube': stacked,
            'metadata': metadata,
            'file_keys': file_keys,
            'frame_indices': frame_indices,
        }

    return result

## Tests

In [ ]:
#| export
def test_assemble_frames():
    """Test frame assembly with heterogeneous dimensions and different interpolation modes."""
    # Create mock frames with different sizes
    frames_small = np.random.randn(40, 100, 5)  # 5 files, small
    frames_large = np.random.randn(40, 200, 3)  # 3 files, large
    frames_dict = {
        'file_small': frames_small,
        'file_large': frames_large
    }

    dims_df = get_frame_dimensions(frames_dict)
    assert list(dims_df.columns) == ['n_files', 'n_pixels', 'n_spectrals']
    assert dims_df.loc['file_small']['n_files'] == 5
    assert dims_df.loc['file_large']['n_pixels'] == 200
    assert dims_df.loc['file_small']['n_spectrals'] == 40

    # Test upsampling with linear interpolation
    cube_up, meta_up = assemble_frames(frames_dict, interp_mode='up', order=1)
    assert cube_up.shape == (40, 200, 8), f"Expected (40, 200, 8), got {cube_up.shape}"
    assert meta_up['interpolation_mode'] == 'up'
    assert meta_up['interpolation_order'] == 1
    assert meta_up['n_temporal_samples'] == 8

    # Test downsampling with cubic interpolation
    cube_down, meta_down = assemble_frames(frames_dict, interp_mode='down', order=3)
    assert cube_down.shape == (40, 100, 8), f"Expected (40, 100, 8), got {cube_down.shape}"
    assert meta_down['interpolation_mode'] == 'down'
    assert meta_down['interpolation_order'] == 3

    # Test no-interp mode (should fail with heterogeneous dims)
    try:
        cube_none, _ = assemble_frames(frames_dict, interp_mode='match')
        assert False, "Should have raised ValueError"
    except ValueError as e:
        assert "heterogeneous" in str(e)

    # Test uniform frames (no interpolation needed)
    frames_uniform = {
        'file1': np.random.randn(40, 100, 5),
        'file2': np.random.randn(40, 100, 3)
    }
    cube_uniform, meta_uniform = assemble_frames(frames_uniform, interp_mode='match')
    assert cube_uniform.shape == (40, 100, 8)
    assert meta_uniform['interpolation_mode'] == 'none'
    assert meta_uniform['interpolation_order'] == 1

    print("✓ All assembly tests passed")

## ADR-007 Tests: RAW Axis Normalization

In [ ]:
def test_normalize_raw_axis_order():
    """Test RAW level axis normalization per ADR-007."""
    # RAW data comes as (frames, pixels, spectrals) = (62415, 100, 40)
    raw_arr = np.random.randn(62415, 100, 40)

    # Normalize to (spectrals, pixels, frames)
    normalized = normalize_raw_axis_order(raw_arr)

    # Expected: (40, 100, 62415)
    assert normalized.shape == (40, 100, 62415), f"Expected (40, 100, 62415), got {normalized.shape}"

    # Verify data integrity (sample check)
    assert np.allclose(raw_arr[0, 0, 0], normalized[0, 0, 0])

    print("✓ ADR-007 axis normalization test passed")

In [ ]:
# Test the normalization
test_normalize_raw_axis_order()